# [ Computer Vision ] [ CSE445 ] [ Assignment-2 ]

**Dataset Name:** IoTKITs<br>
**Source Link:** https://data.mendeley.com/datasets/x5thzmkxhy/1

## Configuration

In [ ]:
!pip install ultralytics --quiet

In [ ]:
import os
import glob
import json
import shutil
import random
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# inputs
partition_dir = "/kaggle/input/notebooks/Mrpaul0007/group-k-partb-00-partition"
yolo_root     = os.path.join(partition_dir, "yolo_rho20")
split_dir     = os.path.join(partition_dir, "splits")

# SimCLR checkpoint
SSL_CKPT = "/kaggle/input/notebooks/Mrpaul0007/self-supervised-learning-nb-1-simclr-pretrain/simclr_backbone.pt"

working_dir = "/kaggle/working/"
os.makedirs(working_dir, exist_ok=True)

# fine-tune
METHOD    = "SimCLR"
BEST_YAML = "yolo12s.yaml"
RHO       = 0.20
SEED      = 42

IMGSZ     = 640
BATCH     = 16
EPOCHS    = 200
OPTIMIZER = "auto"
LR0       = 0.01
LRF       = 0.01
CONF      = 0.25
IOU_NMS   = 0.70

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 0 if torch.cuda.is_available() else "cpu"

print("====== Configuration ======\n")
print("Method            :", METHOD)
print("Architecture      :", BEST_YAML)
print("SSL checkpoint    :", SSL_CKPT)
print("Partition dir     :", partition_dir)
print("Label fraction rho:", RHO)
print("Sampling seed     :", SEED)
print("Image size        :", IMGSZ)
print("Batch size        :", BATCH)
print("Epochs            :", EPOCHS)
print("Optimizer         :", OPTIMIZER)
print("LR0 / LRF         :", LR0, "/", LRF)
print("Conf / NMS IoU    :", CONF, "/", IOU_NMS)
print("Device            :", device)

## Data Yaml

In [ ]:
with open(os.path.join(yolo_root, "data.yaml")) as f:
    print("Original data.yaml:\n" + f.read())

with open(os.path.join(yolo_root, "data.yaml")) as f:
    for line in f:
        if line.startswith("names:"):
            class_names = json.loads(line.split("names:", 1)[1].strip())
        if line.startswith("nc:"):
            NC = int(line.split(":", 1)[1].strip())

DATA_YAML = os.path.join(working_dir, "data.yaml")
with open(DATA_YAML, "w") as f:
    f.write(f"path: {yolo_root}\n"
            f"train: images/train\n"
            f"val: images/val\n"
            f"test: images/test\n"
            f"nc: {NC}\n"
            f"names: {json.dumps(class_names)}\n")

print("Rewritten data.yaml:", DATA_YAML)
print("Classes:", NC, class_names)

## Subset Verification

In [ ]:
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

def stems(split):
    d = os.path.join(yolo_root, "images", split)
    out = set()
    for f in os.listdir(d):
        if f.lower().endswith(IMG_EXT):
            s = f.split("__", 1)[1] if "__" in f else f
            out.add(os.path.splitext(s)[0])
    return out

def instances(split):
    d = os.path.join(yolo_root, "labels", split)
    n = 0
    for f in os.listdir(d):
        if f.endswith(".txt"):
            with open(os.path.join(d, f)) as fh:
                n += sum(1 for line in fh if line.strip())
    return n

def pool_stems():
    d = os.path.join(partition_dir, "ssl_pool_images")
    out = set()
    for f in os.listdir(d):
        if f.lower().endswith(IMG_EXT):
            s = f.split("__", 1)[1] if "__" in f else f
            out.add(os.path.splitext(s)[0])
    return out

tr, va, te = stems("train"), stems("val"), stems("test")
pool = pool_stems()
N = len(pool) + len(va) + len(te)

print(f"Labelled subset D_{RHO} (seed = {SEED})\n")
print(f"{'split':<22}{'images':>8}{'instances':>12}{'% of |D|':>11}")
print(f"{'train (D_0.20)':<22}{len(tr):>8}{instances('train'):>12}{100*len(tr)/N:>10.1f}%")
print(f"{'validation':<22}{len(va):>8}{instances('val'):>12}{100*len(va)/N:>10.1f}%")
print(f"{'test holdout':<22}{len(te):>8}{instances('test'):>12}{100*len(te)/N:>10.1f}%")
print(f"{'SSL pool':<22}{len(pool):>8}{'0 (discarded)':>12}{100*len(pool)/N:>10.1f}%")
print(f"\n|D| = {N}")

print("\nDisjointness checks")
checks = [
    ("train  subset-of ssl_pool", tr - pool),
    ("test   ∩ ssl_pool",         te & pool),
    ("val    ∩ ssl_pool",         va & pool),
    ("test   ∩ train",            te & tr),
    ("val    ∩ train",            va & tr),
]
for name, s in checks:
    print(f"  {name:<28}{len(s):>5}   {'PASS' if not s else 'FAIL'}")
    assert not s, "LEAKAGE: " + name

## Weight Surgery

In [ ]:
from ultralytics import YOLO

det = YOLO(BEST_YAML)
model_sd = det.model.state_dict()

ssl_sd = torch.load(SSL_CKPT, map_location="cpu")
print("SSL checkpoint tensors:", len(ssl_sd))

loaded, skipped = {}, []
for k, v in ssl_sd.items():
    tk = "model." + k
    if tk in model_sd and model_sd[tk].shape == v.shape:
        loaded[tk] = v
    else:
        skipped.append(k)

new_sd = dict(model_sd)
new_sd.update(loaded)
det.model.load_state_dict(new_sd)

SSL_INIT = os.path.join(working_dir, "simclr_init.pt")
torch.save({"model": det.model, "train_args": {}, "epoch": -1}, SSL_INIT)
det = YOLO(SSL_INIT)

def _verify(trainer):
    w = trainer.model.model[0].conv.weight.detach().float().cpu()
    print(">>> backbone == SSL ckpt:", torch.allclose(w, ssl_sd["0.conv.weight"]))
det.add_callback("on_train_start", _verify)

# Report based on layer-index
def layer_idx(key):
    parts = key.split(".")
    return int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else -1

ssl_layers  = sorted({layer_idx(k) for k in loaded})
all_layers  = sorted({layer_idx(k) for k in model_sd if layer_idx(k) >= 0})
rand_layers = [i for i in all_layers if i not in ssl_layers]

print("\n====== Weight surgery report ======")
print(f"Tensors initialised from SSL   : {len(loaded)} / {len(model_sd)}")
print(f"Tensors left randomly init     : {len(model_sd) - len(loaded)}")
print(f"Shape-mismatch / unused in ckpt: {len(skipped)}")
print(f"\nSSL-initialised layer indices  : {ssl_layers}")
print(f"Randomly-initialised layers    : {rand_layers}   (neck + detection head)")
print("\nSaved SSL-init checkpoint      :", SSL_INIT)

print("\nPer-tensor mapping (first 20 SSL-loaded):")
for k in list(loaded)[:20]:
    print(f"  SSL   {k:<50} {tuple(loaded[k].shape)}")
print("\nPer-tensor mapping (first 10 random-init):")
for k in [k for k in model_sd if k not in loaded][:10]:
    print(f"  RAND  {k:<50} {tuple(model_sd[k].shape)}")

## Fine-Tune, rho = 0.20

In [ ]:
FT_ARGS = dict(
    data=DATA_YAML, 
    epochs=EPOCHS, 
    imgsz=IMGSZ, 
    batch=BATCH,
    optimizer=OPTIMIZER, 
    lr0=LR0, 
    lrf=LRF, 
    seed=SEED,
    device=device, 
    project=working_dir, 
    exist_ok=True,
    plots=True,
)

r_ssl = det.train(name="simclr_rho20", **FT_ARGS)
ssl_dir = os.path.join(working_dir, "simclr_rho20")
print("Run dir:", ssl_dir)

### Demo inference on one validation image

In [ ]:
best_ssl = os.path.join(ssl_dir, "weights", "best.pt")
demo_model = YOLO(best_ssl)

demo_img = sorted(glob.glob(os.path.join(yolo_root, "images", "val", "*")))[0]
res = demo_model.predict(demo_img, imgsz=IMGSZ, conf=CONF, iou=IOU_NMS, device=device, verbose=False)[0]

plt.figure(figsize=(8, 8))
plt.imshow(res.plot()[:, :, ::-1])
plt.title("Demo inference — validation image")
plt.axis("off")
plt.show()

print("Image :", os.path.basename(demo_img))
print("Boxes :", len(res.boxes))

## Baselines, rho = 0.20

In [ ]:
rand_dir = "/kaggle/input/notebooks/Mrpaul0007/self-supervised-learning-nb-0-baseline/base_random"
coco_dir = "/kaggle/input/notebooks/Mrpaul0007/self-supervised-learning-nb-0-baseline/base_coco"

print("Reusing baseline checkpoints (no retraining):")
print("Random-init dir:", rand_dir)
print("COCO-pretrained dir:", coco_dir)

## Evaluation — validation + test

In [ ]:
def evaluate(run_dir, tag):
    w = os.path.join(run_dir, "weights", "best.pt")
    m = YOLO(w)
    rows = {}
    for split in ["val", "test"]:
        r = m.val(data=DATA_YAML, split=split, imgsz=IMGSZ, batch=BATCH,
                  conf=0.001, iou=IOU_NMS, device=device,
                  project=working_dir, name=f"{tag}_{split}_eval",
                  exist_ok=True, plots=True, verbose=False)
        rows[split] = dict(
            precision=float(r.box.mp), recall=float(r.box.mr),
            mAP50=float(r.box.map50), mAP50_95=float(r.box.map),
            save_dir=str(r.save_dir),
        )
    return rows

results = {
    "SimCLR (SSL init)": evaluate(ssl_dir,  "simclr"),
    "Random init":       evaluate(rand_dir, "randinit"),
    "COCO-pretrained":   evaluate(coco_dir, "coco"),
}

print(f"\n{'initialisation':<22}{'split':<7}{'P':>8}{'R':>8}{'mAP50':>9}{'mAP50-95':>11}")
for name, rows in results.items():
    for split, v in rows.items():
        print(f"{name:<22}{split:<7}{v['precision']:>8.3f}{v['recall']:>8.3f}"
              f"{v['mAP50']:>9.3f}{v['mAP50_95']:>11.3f}")

## Qualitative predictions

In [ ]:
qual_model = YOLO(best_ssl)
test_imgs = sorted(glob.glob(os.path.join(yolo_root, "images", "test", "*")))
random.Random(SEED).shuffle(test_imgs)
picks = test_imgs[:8]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, p in zip(axes.ravel(), picks):
    r = qual_model.predict(p, imgsz=IMGSZ, conf=CONF, iou=IOU_NMS, device=device, verbose=False)[0]
    n_gt = 0
    lp = os.path.join(yolo_root, "labels", "test", os.path.splitext(os.path.basename(p))[0] + ".txt")
    if os.path.exists(lp):
        with open(lp) as fh:
            n_gt = sum(1 for line in fh if line.strip())
    ax.imshow(r.plot()[:, :, ::-1])
    ax.set_title(f"pred={len(r.boxes)}  gt={n_gt}", fontsize=10)
    ax.axis("off")

plt.suptitle("SimCLR @ rho=0.20 — qualitative test predictions")
plt.tight_layout()
plt.savefig(os.path.join(working_dir, "simclr_qualitative.png"), dpi=150)
plt.show()

## Save Results

In [ ]:
import pandas as pd

rows = []
for label, key in [("random_init",     "Random init"),
                   ("coco_pretrained", "COCO-pretrained"),
                   ("simclr",          "SimCLR (SSL init)")]:
    v = results[key]["test"]
    rows.append({
        "model":     label,
        "rho":       RHO,
        "precision": v["precision"],
        "recall":    v["recall"],
        "map50":     v["mAP50"],
        "map50_95":  v["mAP50_95"],
    })

df = pd.DataFrame(rows)
print(df)

csv_path = os.path.join(working_dir, "simclr_result.csv")
df.to_csv(csv_path, index=False)
print()
print("Saved:", csv_path)